In [ ]:
import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn as nn

REVERSE_CLASS_MAP = {
    0: 'Human', 
    1: 'OpenAI', 
    2: 'Google', 
    3: 'Meta', 
    4: 'Anthropic'
}

# 2. Recreate your PyTorch Model Architecture
class TextoClassificadorDNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(TextoClassificadorDNN, self).__init__()
        self.net = nn.Sequential(
            nn.Dropout(0.5), 
            
            nn.Linear(input_size, 32),
            nn.ReLU(),
            
            nn.Dropout(0.5),
            
            nn.Linear(32, num_classes) 
        )

    def forward(self, x):
        return self.net(x)

# 3. Load the test dataset
df_teste = pd.read_csv('subm2.csv', sep=';')

textos_teste = df_teste['Text'].fillna('')

# 4. Load Vectorizer & Transform
with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

X_teste_tabular = vectorizer.transform(textos_teste).toarray().astype(np.float32)

# 5. Load the PyTorch Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = X_teste_tabular.shape[1]

# Initialize model and load weights
modelo_pytorch = TextoClassificadorDNN(input_size=input_dim, num_classes=5).to(device)
modelo_pytorch.load_state_dict(torch.load('melhor_modelo_pytorch.pth', map_location=device, weights_only=True))
modelo_pytorch.eval()

# 6. Prediction
X_tensor = torch.tensor(X_teste_tabular).to(device)
with torch.no_grad():
    outputs = modelo_pytorch(X_tensor)
    _, previsoes_idx = torch.max(outputs, 1)

previsoes_idx = previsoes_idx.cpu().numpy()

# 7. Map back to strings
df_teste['Label'] = [REVERSE_CLASS_MAP[idx] for idx in previsoes_idx]
df_submissao = df_teste[['ID', 'Label']]
nome_ficheiro_saida = 'subm2-g6-MIA-B.csv'
df_submissao.to_csv(nome_ficheiro_saida, index=False, sep=';')

print(f"Sucesso! Ficheiro {nome_ficheiro_saida} gerado.")
print("\nNova Distribuição de Previsões (PyTorch):")
print(df_submissao['Label'].value_counts())